In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
file_path = "./data_1.csv"
good_df_isobaric_filtered = pd.read_csv(file_path)
good_df_isobaric_filtered['task_type'] = 'isothermal'

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
file_path = "./data_001.xlsx"
good_df_isobaric_filtered = pd.read_excel(file_path)
good_df_isobaric_filtered['task_type'] = 'isothermal'

In [2]:
good_df_isobaric_filtered

,ID,Type,Compound_1,Compound_2,T_K,P_kPa,x,y,SMILES_1,Class_1,SMILES_2,Class_2,task_type
0,123,Isothermal,METHYL ETHYL KETONE,"2,2,4-TRIMETHYLPENTANE",318.15,16.617300,0.0077,0.0403,CCC(=O)C,Ketones,CC(C)CC(C)(C)C,Paraffinic Hydrocarbons,isothermal
1,123,Isothermal,METHYL ETHYL KETONE,"2,2,4-TRIMETHYLPENTANE",318.15,18.637134,0.0386,0.1699,CCC(=O)C,Ketones,CC(C)CC(C)(C)C,Paraffinic Hydrocarbons,isothermal
2,123,Isothermal,METHYL ETHYL KETONE,"2,2,4-TRIMETHYLPENTANE",318.15,20.890282,0.0738,0.2861,CCC(=O)C,Ketones,CC(C)CC(C)(C)C,Paraffinic Hydrocarbons,isothermal
3,123,Isothermal,METHYL ETHYL KETONE,"2,2,4-TRIMETHYLPENTANE",318.15,21.614223,0.0852,0.3184,CCC(=O)C,Ketones,CC(C)CC(C)(C)C,Paraffinic Hydrocarbons,isothermal
4,123,Isothermal,METHYL ETHYL KETONE,"2,2,4-TRIMETHYLPENTANE",318.15,26.481822,0.1669,0.4906,CCC(=O)C,Ketones,CC(C)CC(C)(C)C,Paraffinic Hydrocarbons,isothermal
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1970,4595,Isothermal,O-XYLENE,METHYLCYCLOHEXANE,348.15,17.264000,0.8477,0.5167,CC1=CC=CC=C1C,Aromatics,CC1CCCCC1,Naphtenic Hydrocarbons,isothermal
1971,4595,Isothermal,O-XYLENE,METHYLCYCLOHEXANE,348.15,16.117000,0.8749,0.5698,CC1=CC=CC=C1C,Aromatics,CC1CCCCC1,Naphtenic Hydrocarbons,isothermal
1972,4595,Isothermal,O-XYLENE,METHYLCYCLOHEXANE,348.15,14.543000,0.9110,0.6560,CC1=CC=CC=C1C,Aromatics,CC1CCCCC1,Naphtenic Hydrocarbons,isothermal
1973,4595,Isothermal,O-XYLENE,METHYLCYCLOHEXANE,348.15,13.300000,0.9382,0.7366,CC1=CC=CC=C1C,Aromatics,CC1CCCCC1,Naphtenic Hydrocarbons,isothermal


In [3]:
# 方法1：直接统计不同ID的数量
num_unique_ids = good_df_isobaric_filtered['ID'].nunique()
print(f"不同ID的数量: {num_unique_ids}")

不同ID的数量: 105


In [4]:
# 选取这两列，去除重复行，然后统计行数
unique_pairs = good_df_isobaric_filtered[['Compound_1', 'Compound_2']].drop_duplicates()
num_unique_pairs = len(unique_pairs)
print(f"不同的化合物对 (Compound_1, Compound_2) 数量: {num_unique_pairs}")

不同的化合物对 (Compound_1, Compound_2) 数量: 56


In [ ]:
# 每个化合物对对应的不同 ID 数量
pair_id_counts = (
    good_df_isobaric_filtered
    .groupby(['Compound_1', 'Compound_2'])['ID']
    .nunique()
    .reset_index(name='num_ids')
)
print("\nID数量的分布情况:")
pair_id_counts.head(50)

# 尝试

In [ ]:
import numpy as np
import pandas as pd

df = good_df_isobaric_filtered.copy()
df['pair'] = df['Compound_1'].astype(str) + "|" + df['Compound_2'].astype(str)

# ---------- 1. 构建 pair -> ID 列表 ----------
pair2ids = df.groupby('pair')['ID'].apply(lambda s: sorted(s.unique())).to_dict()

# 拆成"单 ID 化合对"和"多 ID 化合对"
single_pairs = {p: ids for p, ids in pair2ids.items() if len(ids) == 1}
multi_pairs  = {p: ids for p, ids in pair2ids.items() if len(ids) >= 2}

print(f"单 ID 化合对数量: {len(single_pairs)}")
print(f"多 ID 化合对数量: {len(multi_pairs)}")

# ---------- 2. 处理单 ID 化合对（按 8:2 划分 ID）----------
# 单 ID 化合对涉及的 ID 集合
single_ids = [ids[0] for ids in single_pairs.values()]

rng = np.random.RandomState(46)
single_ids_shuffled = single_ids.copy()
rng.shuffle(single_ids_shuffled)

n_test_single = int(round(len(single_ids_shuffled) * 0.2))
single_test_ids  = set(single_ids_shuffled[:n_test_single])
single_train_ids = set(single_ids_shuffled[n_test_single:])

print(f"\n单 ID 化合对: 共 {len(single_ids)} 个 ID, "
      f"train={len(single_train_ids)}, test={len(single_test_ids)}")

# ---------- 3. 处理多 ID 化合对（按规则逐个分配）----------
multi_rule = {
    2: (1, 1),
    3: (2, 1),
    4: (3, 1),
    5: (4, 1),
    6: (4, 2),
    7: (5, 2),
}

multi_train_ids = set()
multi_test_ids  = set()

for pair, ids in multi_pairs.items():
    n = len(ids)
    ids_shuffled = ids.copy()
    rng.shuffle(ids_shuffled)

    if n in multi_rule:
        n_train, n_test = multi_rule[n]
    else:
        # 兜底：>7 时按 8:2
        n_test  = max(1, round(n * 0.2))
        n_train = n - n_test

    for i, _id in enumerate(ids_shuffled):
        if i < n_train:
            multi_train_ids.add(_id)
        else:
            multi_test_ids.add(_id)

# 理论上多 ID 化合对的 ID 不会跨两类，但保险起见处理重叠
overlap = multi_train_ids & multi_test_ids
if overlap:
    # 出现冲突时优先放 train（极少发生）
    multi_test_ids -= overlap

print(f"多 ID 化合对: train ID={len(multi_train_ids)}, test ID={len(multi_test_ids)}")

# ---------- 4. 合并两部分划分结果 ----------
train_ids = single_train_ids | multi_train_ids
test_ids  = single_test_ids  | multi_test_ids

# 防止 ID 同时出现在两边（不该发生，但保险）
conflict = train_ids & test_ids
if conflict:
    test_ids -= conflict   # 冲突时归 train

# ---------- 5. 生成最终 train_df / test_df ----------
train_df = df[df['ID'].isin(train_ids)].copy()
test_df  = df[df['ID'].isin(test_ids)].copy()

# 移除辅助列
train_df = train_df.drop(columns=['pair'])
test_df  = test_df.drop(columns=['pair'])

# ---------- 6. 统计 ----------
total_ids = df['ID'].nunique()
print("\n" + "=" * 50)
print(f"总 ID: {total_ids}, train ID: {len(train_ids)}, test ID: {len(test_ids)}")
print(f"train 行数: {len(train_df)}, test 行数: {len(test_df)}")
print(f"行数比例: {len(train_df)/len(df):.2f} : {len(test_df)/len(df):.2f}")
print(f"ID 比例: {len(train_ids)/total_ids:.2f} : {len(test_ids)/total_ids:.2f}")

# 化合物对覆盖
train_pairs = set(train_df['Compound_1'].astype(str) + "|" + train_df['Compound_2'].astype(str))
test_pairs  = set(test_df['Compound_1'].astype(str) + "|" + test_df['Compound_2'].astype(str))
print("\n化合物对覆盖:")
print(f"  只在 train: {len(train_pairs - test_pairs)}")
print(f"  只在 test : {len(test_pairs  - train_pairs)}")
print(f"  两边都有 : {len(train_pairs & test_pairs)}")

# 多 ID 化合对是否都两边出现
fail = [p for p in multi_pairs if not (p in train_pairs and p in test_pairs)]
print(f"\n多 ID 化合对未能两边都出现的数量: {len(fail)}")
if fail:
    print("示例:", fail[:5])

In [37]:
from sklearn.model_selection import train_test_split

# 获取所有唯一的 ID
unique_ids = good_df_isobaric_filtered['ID'].unique()

# 将 ID 按 7:3 分成训练集和测试集
train_ids, test_ids = train_test_split(unique_ids, test_size=0.2, random_state=62)

# 根据 ID 划分数据集
train_df = good_df_isobaric_filtered[good_df_isobaric_filtered['ID'].isin(train_ids)]
test_df = good_df_isobaric_filtered[good_df_isobaric_filtered['ID'].isin(test_ids)]

# 打印统计信息
print(f"总数据量: {len(good_df_isobaric_filtered)}")
print(f"总 ID 数量: {len(unique_ids)}")
print("-" * 50)
print(f"训练集数据量: {len(train_df)}")
print(f"训练集 ID 数量: {len(train_ids)}")
print(f"训练集比例: {len(train_df)/len(good_df_isobaric_filtered)*100:.2f}%")
print("-" * 50)
print(f"测试集数据量: {len(test_df)}")
print(f"测试集 ID 数量: {len(test_ids)}")
print(f"测试集比例: {len(test_df)/len(good_df_isobaric_filtered)*100:.2f}%")
print("-" * 50)
print(f"实际比例: {len(train_df)/len(good_df_isobaric_filtered):.2f} : {len(test_df)/len(good_df_isobaric_filtered):.2f}")

总数据量: 1975
总 ID 数量: 105
--------------------------------------------------
训练集数据量: 1600
训练集 ID 数量: 84
训练集比例: 81.01%
--------------------------------------------------
测试集数据量: 375
测试集 ID 数量: 21
测试集比例: 18.99%
--------------------------------------------------
实际比例: 0.81 : 0.19


In [38]:
train_df = (
    train_df
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)
train_df

,ID,Type,Compound_1,Compound_2,T_K,P_kPa,x,y,SMILES_1,Class_1,SMILES_2,Class_2,task_type
0,1506,Isothermal,TOLUENE,N-HEPTANE,313.15,8.439,0.9531,0.8944,CC1=CC=CC=C1,Aromatics,CCCCCCC,Paraffinic Hydrocarbons,isothermal
1,1121,Isothermal,N-OCTANE,2-METHYL-2-BUTANOL,328.15,26.310,0.8070,0.7801,CCCCCCCC,Paraffinic Hydrocarbons,CCC(C)(C)O,Alcohols,isothermal
2,947,Isothermal,N-HEPTANE,N-PENTANOL,358.15,68.020,0.9662,0.9609,CCCCCCC,Paraffinic Hydrocarbons,CCCCCO,Alcohols,isothermal
3,946,Isothermal,N-HEPTANE,N-PENTANOL,348.15,28.530,0.1627,0.7817,CCCCCCC,Paraffinic Hydrocarbons,CCCCCO,Alcohols,isothermal
4,2164,Isothermal,N-HEPTANE,O-XYLENE,348.10,30.970,0.4850,0.8090,CCCCCCC,Paraffinic Hydrocarbons,CC1=CC=CC=C1C,Aromatics,isothermal
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1595,2589,Isothermal,CYCLOHEXANE,TERT-BUTANOL,343.30,83.640,0.2020,0.3910,C1CCCCC1,Naphtenic Hydrocarbons,CC(C)(C)O,Alcohols,isothermal
1596,3261,Isothermal,N-BUTANOL,FURFURAL,368.15,32.708,0.4190,0.7810,CCCCO,Alcohols,C1=COC(=C1)C=O,Aldehydes,isothermal
1597,1915,Isothermal,ISOPROPANOL,DI-N-PROPYL ETHER,303.15,12.754,0.1377,0.2487,CC(C)O,Alcohols,CCCOCCC,Ethers,isothermal
1598,3848,Isothermal,TETRAHYDROFURAN,CYCLOHEXANE,333.15,51.960,0.0000,0.0000,C1CCOC1,Ethers,C1CCCCC1,Naphtenic Hydrocarbons,isothermal


In [39]:
# train_df数据增强
import pandas as pd
# ===========================
# 原始训练集
# ===========================
train_aug = train_df.copy()
# ===========================
# 构造交换后的样本
# ===========================
train_swap = train_df.copy()
# 交换名称
train_swap["Compound_1"], train_swap["Compound_2"] = (
    train_df["Compound_2"],
    train_df["Compound_1"]
)
# 交换SMILES
train_swap["SMILES_1"], train_swap["SMILES_2"] = (
    train_df["SMILES_2"],
    train_df["SMILES_1"]
)
# x,y转换
train_swap["x"] = 1 - train_df["x"]
train_swap["y"] = 1 - train_df["y"]
# 温度、压力、System_ID、System_Key保持不变
# 不需要修改
# ===========================
# 合并
# ===========================
train_aug = pd.concat(
    [train_aug, train_swap],
    ignore_index=True
)
print("Original train:", len(train_df))
print("Augmented train:", len(train_aug))

Original train: 1600
Augmented train: 3200


In [40]:
train_aug = (
    train_aug
    .sample(frac=1, random_state=1)
    .reset_index(drop=True)
)
train_aug

,ID,Type,Compound_1,Compound_2,T_K,P_kPa,x,y,SMILES_1,Class_1,SMILES_2,Class_2,task_type
0,3674,Isothermal,METHYL PROPIONATE,N-HEPTANE,343.15,77.130,0.5077,0.6722,CCC(=O)OC,Esters,CCCCCCC,Paraffinic Hydrocarbons,isothermal
1,3222,Isothermal,ISOBUTYRALDEHYDE,ETHYL ACETATE,313.15,33.290,0.5790,0.4480,CC(C)C=O,Esters,CCOC(=O)C,Aldehydes,isothermal
2,1182,Isothermal,2-METHYL-1-BUTANOL,N-HEPTANE,368.15,94.930,0.1169,0.1126,CCC(C)CO,Paraffinic Hydrocarbons,CCCCCCC,Alcohols,isothermal
3,829,Isothermal,BENZALDEHYDE,ETHYLBENZENE,358.15,13.150,0.5487,0.2076,C1=CC=C(C=C1)C=O,Aromatics,CCC1=CC=CC=C1,Aldehydes,isothermal
4,2589,Isothermal,CYCLOHEXANE,TERT-BUTANOL,343.30,87.760,0.9410,0.7900,C1CCCCC1,Naphtenic Hydrocarbons,CC(C)(C)O,Alcohols,isothermal
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3195,1891,Isothermal,DI-N-PROPYL ETHER,N-PROPANOL,308.15,13.966,0.8538,0.8510,CCCOCCC,Alcohols,CCCO,Ethers,isothermal
3196,1512,Isothermal,P-XYLENE,N-DECANE,313.15,2.114,0.7258,0.9304,CC1=CC=C(C=C1)C,Aromatics,CCCCCCCCCC,Paraffinic Hydrocarbons,isothermal
3197,1182,Isothermal,N-HEPTANE,2-METHYL-1-BUTANOL,368.15,78.410,0.3486,0.7296,CCCCCCC,Paraffinic Hydrocarbons,CCC(C)CO,Alcohols,isothermal
3198,2589,Isothermal,CYCLOHEXANE,TERT-BUTANOL,343.30,74.400,0.1110,0.2640,C1CCCCC1,Naphtenic Hydrocarbons,CC(C)(C)O,Alcohols,isothermal


In [41]:
import json
import pandas as pd
import numpy as np
def fmt_num(x, digits):
    """
    保留固定小数位，并返回字符串
    """
    return f"{float(x):.{digits}f}"
# ============================================================
# 3. DataFrame 转 JSONL 样本列表
# ============================================================
def df_to_jsonl_samples(df):
    """
    将整个 DataFrame 转换成 JSONL 样本列表。
    """
    df = df.copy()
    df.columns = df.columns.str.strip()

    required_cols = [
        "SMILES_1",
        "SMILES_2",
        "P_kPa",
        "x",
        "T_K",
        "y"
    ]

    missing_cols = [col for col in required_cols if col not in df.columns]

    if missing_cols:
        raise ValueError(f"缺少必要列: {missing_cols}")
    # 删除关键列缺失的数据
    df = df.dropna(subset=required_cols).copy()
    samples = []
    for _, row in df.iterrows():
        samples.append(row_to_sample(row))
    return samples
# ============================================================
# 4. 保存 JSONL 文件
# ============================================================
def save_jsonl(samples, save_path):
    """
    保存为 JSONL 文件。
    每一行是一条 JSON 数据。
    """
    with open(save_path, "w", encoding="utf-8") as f:
        for sample in samples:
            f.write(json.dumps(sample, ensure_ascii=False) + "\n")

    print(f"保存完成: {save_path}")
    print(f"样本数: {len(samples)}")
def row_to_sample(row):

    smiles1 = str(row["SMILES_1"]).strip()
    smiles2 = str(row["SMILES_2"]).strip()

    x1 = fmt_num(row["x"], 3)
    y1 = fmt_num(row["y"], 3)

    T = fmt_num(row["T_K"], 2)

    # bar → kPa
    P = fmt_num(row["P_kPa"], 3)

    input_text = f"""Task: Predict the vapor-liquid equilibrium state of a binary mixture under isothermal conditions.
Given:
Component 1 SMILES: {smiles1}
Component 2 SMILES: {smiles2}
Operating temperature: {T} K
Liquid-phase mole fraction of component 1: {x1}
Please predict:
1. Vapor-phase mole fraction of component 1
2. Equilibrium pressure
"""

    output_text = f"""Vapor-phase mole fraction of component 1: {y1}
Equilibrium pressure: {P} kPa"""

    return {
        "input": input_text,
        "output": output_text
    }

In [44]:
train_samples = df_to_jsonl_samples(train_aug)
save_jsonl(train_samples, "./data/训练_isothermal_62.jsonl")

保存完成: ./data/训练_isothermal_62.jsonl
样本数: 3200


In [45]:
train_samples_df = df_to_jsonl_samples(train_df)
save_jsonl(train_samples_df, "./data/train_isothermal_62.jsonl")
test_samples = df_to_jsonl_samples(test_df)
save_jsonl(test_samples, "./data/test_isothermal_62.jsonl")

保存完成: ./data/train_isothermal_62.jsonl
样本数: 1600
保存完成: ./data/test_isothermal_62.jsonl
样本数: 375
